### Objective

In this notebook, we will use AutoML library to address the NSL-KDD dataset

### 1. Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

#### 1.1 Load dataset

In [ ]:
# import train data
train = pd.read_csv('dataset/train_AutoML_probe.csv')
train.rename(columns={'attack_category': 'attack'}, inplace=True)

# import test data
test = pd.read_csv('dataset/test_AutoML_probe.csv')
test.rename(columns={'attack_category': 'attack'}, inplace=True)

#### 1.2 Feature category

In [ ]:
feature_names = defaultdict(list)
with open('dataset/feature_types.txt', 'r') as f:
    for line in f.readlines():
        feature, category = line.strip().strip('.').split(': ')
        feature_names[category].append(feature)

# continuous features
continuous_features = feature_names['continuous']
continuous_features.remove('num_outbound_cmds')

# binary features
binary_features = ['land', 'logged_in', 'root_shell', 'su_attempted', 'is_host_login', 'is_guest_login']

# nominal features
nominal_features = list(set(feature_names['discrete'])-set(binary_features))

#### 1.3 Ordinal encoding

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()
train[nominal_features] = enc.fit_transform(train[nominal_features])
test[nominal_features] = enc.transform(test[nominal_features])

# Separate feature/label
X_train, y_train = train.iloc[:, :-1].to_numpy(), train.iloc[:, -1].to_numpy()
X_test, y_test = test.iloc[:, :-1].to_numpy(), test.iloc[:, -1].to_numpy()

### 2. ML modeling

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import (
    precision_score, 
    recall_score, 
    accuracy_score, 
    roc_auc_score,
    f1_score,
    matthews_corrcoef
)


def metrics_display(y_test, y_pred, y_pred_proba):

    # Obtain confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Output classification metrics
    tn, fp, fn, tp = cm.ravel()
    
    print(f'ROC_AUC score: {roc_auc_score(y_test, y_pred_proba):.3f}')
    print(f'f1 score: {f1_score(y_test, y_pred):.3f}')
    print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
    print(f'Precision: {precision_score(y_test, y_pred)*100:.2f}%')
    print(f'Detection rate: {recall_score(y_test, y_pred)*100:.2f}%')
    print(f'False alarm rate: {fp / (tn+fp)*100}%')
    print(f'MCC: {matthews_corrcoef(y_test, y_pred):.2f}')
    
    # Display confusion matrix
    # ConfusionMatrixDisplay.from_predictions(y_test, y_pred, values_format='.5g', colorbar=False)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()

#### AutoML with FLAML

In [ ]:
from flaml import AutoML
automl = AutoML()
automl.fit(X_train, y_train, task="classification", time_budget=3600, estimator_list=['xgboost'],
          log_file_name='automl.log', log_type='best')

In [ ]:
y_pred = automl.predict(X_test)
y_pred_proba = automl.predict_proba(X_test)[:, 1]
metrics_display(y_test, y_pred, y_pred_proba)